# RAI Taxonomy v1.0.0 Data Quality Audit

Objective: verify the identity, crosswalk, Physical lock, placement, and open-set invariants of the prepublication 1,726-card release.

Success means all structural assertions pass. A nonzero `needs_taxonomy_decision` count is expected and is reported rather than treated as a failed assertion. This notebook intentionally produces no charts or images.


In [1]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'jupyter-notebook':
    PROJECT_ROOT = PROJECT_ROOT.parents[1]
RELEASE = 'v1.0.0'
DATA = PROJECT_ROOT / 'data' / 'releases' / RELEASE
PUBLIC = PROJECT_ROOT / 'public' / 'data' / 'releases' / RELEASE
REPORT = PROJECT_ROOT / 'reports' / 'validation' / RELEASE

def load(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))


## Plan

1. Confirm expected grain and unique identifiers.
2. Confirm 1,908 crosswalk rows and the 169/13 Physical split.
3. Confirm all 182 Physical placements preserve their source L3.
4. Confirm every unresolved card has a null primary L3.
5. Summarize provisional coverage without interpreting it as validated accuracy.


In [2]:
nodes = load(DATA / 'taxonomy_nodes.json')
registry = load(DATA / 'l4_registry.json')
crosswalk = load(DATA / 'source_crosswalk.json')
locks = load(DATA / 'physical_lock.json')
placements = load(DATA / 'placements.json')
site_cards = load(PUBLIC / 'cards.json')['cards']
validation = load(REPORT / 'validation_summary.json')

profile = {
    'taxonomy_nodes': len(nodes),
    'l4_registry': len(registry),
    'crosswalk_rows': len(crosswalk),
    'physical_locks': len(locks),
    'placements': len(placements),
    'site_cards': len(site_cards),
}
profile


{'taxonomy_nodes': 60,
 'l4_registry': 1726,
 'crosswalk_rows': 1908,
 'physical_locks': 182,
 'placements': 1726,
 'site_cards': 1726}

In [3]:
expected_l4 = [f'RAI4-{n:04d}' for n in range(1, 1727)]
assert Counter(node['level'] for node in nodes) == {0: 1, 1: 3, 2: 6, 3: 50}
assert [card['l4_id'] for card in registry] == expected_l4
assert len({card['l4_id'] for card in registry}) == 1726
assert all('primary_l3_id' not in card for card in registry)
assert len(crosswalk) == 1908
assert Counter(row['relationship'] for row in crosswalk if row['source_system'] == 'physical_182') == {
    'exact_id': 169,
    'explicit_alias': 13,
}
assert len(placements) == 1726 == len({row['l4_id'] for row in placements})
assert len(site_cards) == 1726 == len({row['l4_id'] for row in site_cards})
'core identity and crosswalk assertions passed'


'core identity and crosswalk assertions passed'

In [4]:
placement_by_l4 = {row['l4_id']: row for row in placements}
assert len(locks) == 182
assert Counter(row['legacy_l2_id'] for row in locks) == {'P2': 91, 'I2': 62, 'S2': 29}
assert all(
    placement_by_l4[row['l4_id']]['assignment_status'] == 'locked_physical'
    and placement_by_l4[row['l4_id']]['primary_l3_id'] == row['new_l3_id']
    for row in locks
)
physical_check = {
    'locked': len(locks),
    'legacy_l3_count': len({row['legacy_l3_id'] for row in locks}),
    'distribution': dict(Counter(row['legacy_l2_id'] for row in locks)),
}
physical_check


{'locked': 182,
 'legacy_l3_count': 24,
 'distribution': {'P2': 91, 'I2': 62, 'S2': 29}}

In [5]:
status_counts = Counter(row['assignment_status'] for row in placements)
needs = [row for row in placements if row['assignment_status'] == 'needs_taxonomy_decision']
proposed = [row for row in placements if row['assignment_status'] == 'algorithm_proposed']
assert all(row['primary_l3_id'] is None for row in needs)
assert all(row['primary_l3_id'] is not None for row in proposed)
assert all(row['legacy_hierarchy_used_as_feature'] is False for row in placements)
assert all(
    len(row.get('frontier_expert_reviews', [])) == 2
    and {review['decision'] for review in row['frontier_expert_reviews']} == {'APPROVE'}
    and all(review['hierarchy_blind'] for review in row['frontier_expert_reviews'])
    for row in proposed
)
placement_summary = {
    'status_counts': dict(status_counts),
    'nonphysical_provisional_coverage': round(len(proposed) / 1544, 4),
    'nonphysical_abstention_rate': round(len(needs) / 1544, 4),
    'confidence_calibrated': False,
}
placement_summary


{'status_counts': {'needs_taxonomy_decision': 1507,
  'algorithm_proposed': 37,
  'locked_physical': 182},
 'nonphysical_provisional_coverage': 0.024,
 'nonphysical_abstention_rate': 0.976,
 'confidence_calibrated': False}

## Result and next gate

Structural QA is complete when the validator reports no failed checks. The current release remains provisional: algorithmic consensus is not a substitute for the planned human gold-set validation. Review the unresolved clusters and decision queue before expanding any definition or adding a new L3.


In [6]:
final_result = {
    'validator_status': validation['status'],
    'failed_checks': validation['failed_checks'],
    'warning_checks': validation['warning_checks'],
    'fifty_l3_sufficiency': load(REPORT / 'coverage_summary.json')['fifty_l3_sufficiency_status'],
}
assert final_result['failed_checks'] == 0
final_result


{'validator_status': 'PASS_WITH_WARNINGS',
 'failed_checks': 0,
 'warning_checks': 2,
 'fifty_l3_sufficiency': 'NOT_DEMONSTRATED'}